# Dockerizing a Django Project


## Goal

In this notebook we package a Django or Django REST Framework project with Docker.

We focus on a local/development Docker setup first:

```text
Django container + PostgreSQL container
```

Production-style Nginx, Gunicorn, Redis, and Celery come in the next notebook.


## Why Docker for Django?

Docker helps Django projects because:

- Teammates can run the same environment.
- PostgreSQL can run without installing it directly on the host OS.
- Python/system dependencies are documented in Docker files.
- Local, staging, and production environments can become more similar.
- Dependency conflicts are reduced.

Docker does not replace understanding Django settings, databases, static files, or deployment. It packages them.


## Prepare the Repository

A clean Django project should have:

```text
project/
    manage.py
    requirements.txt
    .env.example
    .gitignore
    project_name/
        settings.py
        urls.py
        wsgi.py
        asgi.py
```

For Docker, we add:

```text
Dockerfile
.dockerignore
compose.yaml
```

`requirements.txt` should include the packages needed to run the project.


## Development Dockerfile for Django

```dockerfile
FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

WORKDIR /app

COPY requirements.txt /app/
RUN pip install --no-cache-dir -r requirements.txt

COPY . /app/

EXPOSE 8000

CMD ["python", "manage.py", "runserver", "0.0.0.0:8000"]
```

Explanation:

| Line | Meaning |
|------|---------|
| `PYTHONDONTWRITEBYTECODE=1` | Avoid writing `.pyc` files. |
| `PYTHONUNBUFFERED=1` | Print logs immediately. |
| `WORKDIR /app` | Project lives in `/app`. |
| `COPY requirements.txt` then install | Better Docker cache. |
| `COPY . /app/` | Copy source code. |
| `CMD ... runserver ...` | Development command. |

For production, we should use Gunicorn instead of `runserver`.


## `.dockerignore` for Django

```dockerignore
.git/
__pycache__/
*.pyc
*.pyo
*.pyd
.Python
venv/
.env
.env.*
!.env.example
.pytest_cache/
.mypy_cache/
db.sqlite3
static_collected/
media/
```

Why?

- Avoid copying local virtualenv.
- Avoid copying secrets.
- Avoid copying generated files.
- Keep image builds faster and safer.


## Django Settings for Docker Database

Inside Compose, Django connects to PostgreSQL using the database service name.

Example:

```dotenv
DATABASE_URL=postgres://user:password@db:5432/mydatabase
```

Host is `db` because the Compose service is named `db`.

In `settings.py`, use environment variables:

```python
import environ

env = environ.Env()

DATABASES = {
    "default": env.db("DATABASE_URL")
}
```

For class examples, you can use `django-environ` or any similar settings approach.


## Compose File: Django + PostgreSQL

```yaml
services:
  web:
    build: .
    command: python manage.py runserver 0.0.0.0:8000
    ports:
      - "8000:8000"
    volumes:
      - .:/app
    depends_on:
      - db
    environment:
      DJANGO_DEBUG: "True"
      DATABASE_URL: postgres://user:password@db:5432/mydatabase

  db:
    image: postgres:15
    environment:
      POSTGRES_USER: user
      POSTGRES_PASSWORD: password
      POSTGRES_DB: mydatabase
    volumes:
      - pgdata:/var/lib/postgresql/data

volumes:
  pgdata:
```

This is a development setup. The bind mount `.:/app` lets code changes appear inside the container.


## Build and Run

Build images and start containers:

```bash
docker compose up --build
```

Run in background:

```bash
docker compose up --build -d
```

Open:

```text
http://localhost:8000
```

View logs:

```bash
docker compose logs -f web
```


## Migrations, Superuser, Shell

Run migrations inside the web container:

```bash
docker compose exec web python manage.py migrate
```

Create superuser:

```bash
docker compose exec web python manage.py createsuperuser
```

Open Django shell:

```bash
docker compose exec web python manage.py shell
```

Run tests if the project has tests:

```bash
docker compose exec web python manage.py test
```


## Static and Media in Development Docker

For development, Django can serve static files when `DEBUG=True`.

Media files may be stored in a mounted folder:

```yaml
services:
  web:
    volumes:
      - .:/app
      - media_data:/app/media

volumes:
  media_data:
```

For production, static/media should be handled more carefully, usually with Nginx and named volumes or object storage.


## Waiting for the Database

`depends_on` controls startup order, but it does not always mean PostgreSQL is ready to accept connections.

If Django starts too early, you may see database connection errors.

Common solutions:

- Restart the web container after DB becomes ready.
- Add a wait script.
- Use healthchecks.
- Make Django retry DB connections on startup.

For classroom development, rerunning migrations after DB is ready is often enough.


## Stop and Clean Up

Stop containers:

```bash
docker compose down
```

Stop and remove volumes:

```bash
docker compose down -v
```

Warning: `down -v` deletes database volume data.

Rebuild from scratch:

```bash
docker compose build --no-cache
```


## Summary

- Docker can package a Django project and its dependencies.
- Compose can run Django and PostgreSQL together.
- In Compose, use service names like `db` instead of `localhost`.
- Use volumes for PostgreSQL data and development source mounts.
- Use `docker compose exec web ...` for Django management commands.
- Development Docker can use `runserver`; production should use Gunicorn and Nginx.
